# Customer Churn Prediction using AI-Driven Data Analytics

**IBM SkillsBuild Data Analytics with AI — Academic Internship 2026**
**Organized by BharatCares, in association with AICTE**

**Author:** AYUSH
**Project:** Customer Churn Prediction using AI-Driven Data Analytics

---

## 1. Problem Statement
Customer churn (customers discontinuing a service) directly affects the revenue and growth of subscription-based businesses such as telecom companies. This project applies data analytics and machine learning (AI) techniques to:
- Explore and understand the factors that drive customer churn
- Build predictive models that flag customers likely to churn
- Generate insights that a business can act on to improve retention

## 2. Dataset
This project follows the schema of the widely-used **IBM Telco Customer Churn** dataset:
Kaggle link: https://www.kaggle.com/datasets/blastchar/telco-customer-churn

Since this notebook is built to run anywhere without requiring a Kaggle account or internet download, **Section 3** generates a synthetic dataset of 2,000 customers with the same columns and realistic statistical relationships as the original dataset (e.g., month-to-month contracts and low tenure increase churn risk).

> **To use the real dataset instead:** download `WA_Fn-UseC_-Telco-Customer-Churn.csv` from the Kaggle link above, place it in the same folder as this notebook, and replace the code in Section 3 with:
> ```python
> df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")
> ```
> The rest of the notebook (Sections 4 onward) will run unchanged on the real data, since column names match.

## 2. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)
%matplotlib inline

## 3. Dataset Creation

Generates a synthetic Telco-style customer dataset. Churn probability is derived from realistic risk factors (contract type, tenure, internet service, payment method) so the dataset has genuine, learnable patterns — the same patterns documented in real-world churn studies.

In [ ]:
np.random.seed(42)
n = 2000

gender = np.random.choice(["Male", "Female"], n)
senior_citizen = np.random.choice([0, 1], n, p=[0.84, 0.16])
partner = np.random.choice(["Yes", "No"], n, p=[0.48, 0.52])
dependents = np.random.choice(["Yes", "No"], n, p=[0.30, 0.70])
tenure = np.random.randint(0, 73, n)
phone_service = np.random.choice(["Yes", "No"], n, p=[0.90, 0.10])
multiple_lines = np.random.choice(["Yes", "No", "No phone service"], n, p=[0.42, 0.48, 0.10])
internet_service = np.random.choice(["DSL", "Fiber optic", "No"], n, p=[0.35, 0.44, 0.21])
online_security = np.random.choice(["Yes", "No", "No internet service"], n, p=[0.29, 0.50, 0.21])
tech_support = np.random.choice(["Yes", "No", "No internet service"], n, p=[0.29, 0.50, 0.21])
streaming_tv = np.random.choice(["Yes", "No", "No internet service"], n, p=[0.38, 0.41, 0.21])
contract = np.random.choice(["Month-to-month", "One year", "Two year"], n, p=[0.55, 0.24, 0.21])
paperless_billing = np.random.choice(["Yes", "No"], n, p=[0.59, 0.41])
payment_method = np.random.choice(
    ["Electronic check", "Mailed check", "Bank transfer (automatic)", "Credit card (automatic)"],
    n, p=[0.34, 0.23, 0.22, 0.21]
)
monthly_charges = np.round(np.random.normal(64, 30, n).clip(18, 120), 2)
total_charges = np.round(monthly_charges * tenure + np.random.normal(0, 50, n), 2).clip(0, None)

# Build churn probability from realistic risk factors, then sample the label
risk = (
    0.45 * (contract == "Month-to-month")
    + 0.20 * (internet_service == "Fiber optic")
    + 0.15 * (payment_method == "Electronic check")
    - 0.25 * (tenure > 24)
    - 0.15 * (partner == "Yes")
    + 0.10 * (senior_citizen == 1)
    + np.random.normal(0, 0.15, n)
)
churn_prob = 1 / (1 + np.exp(-(risk * 4 - 1.2)))
churn = np.where(np.random.rand(n) < churn_prob, "Yes", "No")

df = pd.DataFrame({
    "customerID": [f"CUST-{i:05d}" for i in range(n)],
    "gender": gender,
    "SeniorCitizen": senior_citizen,
    "Partner": partner,
    "Dependents": dependents,
    "tenure": tenure,
    "PhoneService": phone_service,
    "MultipleLines": multiple_lines,
    "InternetService": internet_service,
    "OnlineSecurity": online_security,
    "TechSupport": tech_support,
    "StreamingTV": streaming_tv,
    "Contract": contract,
    "PaperlessBilling": paperless_billing,
    "PaymentMethod": payment_method,
    "MonthlyCharges": monthly_charges,
    "TotalCharges": total_charges,
    "Churn": churn,
})

df.to_csv("Telco-Customer-Churn.csv", index=False)
print("Dataset shape:", df.shape)
df.head()

## 4. Data Understanding & Cleaning

In [ ]:
print("Data types:\n", df.dtypes)
print("\nMissing values per column:\n", df.isnull().sum())
print("\nChurn distribution:\n", df["Churn"].value_counts(normalize=True).round(3))
df.describe()

## 5. Exploratory Data Analysis (EDA)

We visualize the churn distribution and how it relates to tenure, contract type, and monthly charges.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
df["Churn"].value_counts().plot(kind="bar", ax=axes[0], color=["#2E86AB", "#E63946"])
axes[0].set_title("Churn Distribution")
axes[0].set_xlabel("Churn")
axes[0].set_ylabel("Number of Customers")

sns.boxplot(data=df, x="Churn", y="tenure", ax=axes[1])
axes[1].set_title("Tenure vs Churn")
plt.tight_layout()
plt.savefig("eda_churn_tenure.png", dpi=120)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.countplot(data=df, x="Contract", hue="Churn", ax=axes[0])
axes[0].set_title("Churn by Contract Type")
axes[0].tick_params(axis="x", rotation=20)

sns.boxplot(data=df, x="Churn", y="MonthlyCharges", ax=axes[1])
axes[1].set_title("Monthly Charges vs Churn")
plt.tight_layout()
plt.savefig("eda_contract_charges.png", dpi=120)
plt.show()

In [ ]:
corr_df = df.copy()
for col in ["Partner", "Dependents", "PhoneService", "PaperlessBilling", "Churn"]:
    corr_df[col] = corr_df[col].map({"Yes": 1, "No": 0})

numeric_cols = ["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges", "Partner", "Dependents", "Churn"]
plt.figure(figsize=(7, 6))
sns.heatmap(corr_df[numeric_cols].corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.savefig("eda_correlation_heatmap.png", dpi=120)
plt.show()

**Observations:**
- Customers on **month-to-month contracts** churn at a noticeably higher rate than those on one/two-year contracts.
- **Lower tenure** customers (newer customers) are more likely to churn.
- **Higher monthly charges** correlate with a higher probability of churn.

## 6. Data Preprocessing

Categorical variables are label-encoded, features and target are split, and numeric features are scaled for the logistic regression model.

In [ ]:
model_df = df.drop(columns=["customerID"]).copy()

label_encoders = {}
categorical_cols = model_df.select_dtypes(include=["object", "str"]).columns.drop("Churn")
for col in categorical_cols:
    le = LabelEncoder()
    model_df[col] = le.fit_transform(model_df[col])
    label_encoders[col] = le

target_le = LabelEncoder()
model_df["Churn"] = target_le.fit_transform(model_df["Churn"])  # No=0, Yes=1

X = model_df.drop(columns=["Churn"])
y = model_df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training set:", X_train.shape, " Test set:", X_test.shape)

## 7. Model Building (AI Layer)

Two classification models are trained and compared:
- **Logistic Regression** — a simple, interpretable linear baseline
- **Random Forest** — an ensemble model that usually captures non-linear patterns better

In [ ]:
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train_scaled, y_train)

rf = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42)
rf.fit(X_train, y_train)

results = {}
for name, model, data_test in [
    ("Logistic Regression", log_reg, X_test_scaled),
    ("Random Forest", rf, X_test),
]:
    preds = model.predict(data_test)
    proba = model.predict_proba(data_test)[:, 1]
    results[name] = {
        "accuracy": accuracy_score(y_test, preds),
        "precision": precision_score(y_test, preds),
        "recall": recall_score(y_test, preds),
        "f1": f1_score(y_test, preds),
        "roc_auc": roc_auc_score(y_test, proba),
        "preds": preds,
        "proba": proba,
    }

results_df = pd.DataFrame({
    m: {k: v for k, v in r.items() if k not in ("preds", "proba")}
    for m, r in results.items()
}).T
results_df

## 8. Model Evaluation

In [ ]:
best_name = results_df["roc_auc"].idxmax()
best = results[best_name]
print(f"Best model: {best_name}\n")
print(classification_report(y_test, best["preds"], target_names=target_le.classes_))

In [ ]:
cm = confusion_matrix(y_test, best["preds"])
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=target_le.classes_, yticklabels=target_le.classes_)
plt.title(f"Confusion Matrix - {best_name}")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=120)
plt.show()

In [ ]:
plt.figure(figsize=(6, 5))
for name, r in results.items():
    fpr, tpr, _ = roc_curve(y_test, r["proba"])
    plt.plot(fpr, tpr, label=f"{name} (AUC={r['roc_auc']:.2f})")
plt.plot([0, 1], [0, 1], "k--", alpha=0.4)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend()
plt.tight_layout()
plt.savefig("roc_curve.png", dpi=120)
plt.show()

## 9. Feature Importance (Explainable AI)

Understanding *why* the model predicts churn is as important as the prediction itself — this is what makes the model actionable for the business.

In [ ]:
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
plt.figure(figsize=(8, 6))
importances.head(10).sort_values().plot(kind="barh", color="#2E86AB")
plt.title("Top 10 Feature Importances (Random Forest)")
plt.tight_layout()
plt.savefig("feature_importance.png", dpi=120)
plt.show()

importances.head(10)

## 10. Conclusion & Business Insights

- **Contract type, total/monthly charges, and tenure** are the strongest predictors of churn.
- Customers on **month-to-month contracts** with **high monthly charges** and **short tenure** are the highest-risk segment — retention teams should prioritize this group with loyalty offers or incentives to move to annual contracts.
- The **Random Forest** model outperformed Logistic Regression on ROC-AUC, indicating churn is driven by non-linear interactions between features.
- **Next steps:** train on the real IBM Telco dataset (linked in Section 2), tune hyperparameters with cross-validation, and experiment with gradient-boosted models (XGBoost/LightGBM) for further accuracy gains.

---
*Project submitted as part of the AICTE | IBM SkillsBuild Data Analytics with AI Internship 2026, in association with BharatCares.*